In [1]:
import os
import glob
import tgt
import pandas as pd
import pickle
import tqdm
import warnings
warnings.simplefilter("ignore")


import sys
sys.path.insert(0, "../../..")

from utils import data_processing

In [2]:
with open(
    "../../pronunciation_counts/pronunciation_dict_fr.pkl",
    "rb"
) as f:
    pronunciation_dict = pickle.load(f)

In [3]:
DATASET_PATH = "/Volumes/Seagate Backup Plus Drive/ultrasound_llm/dataset/"
identifiers = []
for f in tqdm.tqdm(glob.glob(os.path.join(DATASET_PATH,"dataset/*_frames.npy"))):
    if os.path.isfile(f.replace("_frames", "_fps")) and os.path.isfile(
        f.replace("_frames.npy", ".TextGrid")
    ):
        identifier = os.path.basename(f).replace("_frames.npy", "")
        if "aud" in identifier or "spo" in identifier:
            identifiers += [identifier]

identifiers = [i for i in identifiers if i[3]=="e"]
tgrds = [(tgt.read_textgrid(DATASET_PATH+"dataset/"+i+".TextGrid"), i) for i in tqdm.tqdm(identifiers)]

100%|██████████| 7152/7152 [00:01<00:00, 6250.46it/s]


In [4]:
dfs_final = []

for tgrd, identifier in tqdm.tqdm(tgrds):
    

    df_final = data_processing.alignment_to_dataframe(
        tgrd,
        lang="be",
        pronunciation_dict=pronunciation_dict
    )
    df_final["identifier"] = identifier
    df_final["ac_f_folder"] = os.path.join(DATASET_PATH, "dataset_audio")
    df_final["art_f_folder"] = os.path.join(DATASET_PATH, "dataset")
    dfs_final += [df_final]

100%|██████████| 7152/7152 [00:09<00:00, 785.62it/s]


In [6]:
df = pd.concat(dfs_final, ignore_index=True)

In [7]:
df.to_csv("datasets/processed_beus.csv", index=False)